# Интеллектуальная система мониторинга динамики стоимости образовательных услуг
*(на примере ЧОУ ВО «Московский университет имени С.Ю. Витте»)*

Данный ноутбук содержит этапы:
1. Загрузка и анализ датасета `data.csv`.
2. Визуализация динамики стоимости обучения.
3. Обучение моделей линейной регрессии по образовательным программам.
4. Прогноз стоимости обучения на будущие годы.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import seaborn as sns
%matplotlib inline


## Загрузка и первичный анализ данных

In [ ]:
df = pd.read_csv('data.csv')
df.head()

In [ ]:
df.info()

Ожидаемая структура датасета:

- `year` — год;
- `university` — наименование вуза;
- `program` — образовательная программа (направление подготовки);
- `form` — форма обучения (например, очная);
- `cost` — стоимость обучения в рублях.

## Базовый статистический анализ

In [ ]:
df.describe(include='all')

## Динамика стоимости обучения в МУИВ по программам

In [ ]:
muiv = df[df['university'].str.contains('МУИВ')]
plt.figure(figsize=(10,6))
sns.lineplot(data=muiv, x='year', y='cost', hue='program', marker='o')
plt.title('Динамика стоимости обучения в МУИВ по программам')
plt.ylabel('Стоимость, руб.')
plt.grid(True)
plt.show()

## Сравнение стоимости обучения по программе «Экономика» между вузами

In [ ]:
eco = df[df['program'] == 'Экономика']
plt.figure(figsize=(10,6))
sns.lineplot(data=eco, x='year', y='cost', hue='university', marker='o')
plt.title('Сравнение стоимости обучения по программе "Экономика"')
plt.ylabel('Стоимость, руб.')
plt.grid(True)
plt.show()

## Обучение моделей линейной регрессии по образовательным программам

In [ ]:
models = {}
for program in df['program'].unique():
    df_prog = df[df['program'] == program]
    X = df_prog[['year']]
    y = df_prog['cost']
    model = LinearRegression()
    model.fit(X, y)
    models[program] = model
    print(f'Программа: {program}, коэффициент наклона: {model.coef_[0]:.2f}, сдвиг: {model.intercept_:.2f}')

## Функция для прогноза стоимости обучения

In [ ]:
def predict_cost(program, year):
    if program not in models:
        raise ValueError(f'Нет модели для программы: {program}')
    model = models[program]
    X = np.array([[year]])
    return float(model.predict(X)[0])

predict_cost('Экономика', 2026)

## Прогноз стоимости обучения на несколько лет вперёд

In [ ]:
future_years = list(range(2025, 2031))
program = 'Экономика'
preds = []
for y in future_years:
    preds.append({'year': y, 'program': program, 'pred_cost': predict_cost(program, y)})
pred_df = pd.DataFrame(preds)
pred_df

## Визуализация фактических и прогнозных значений для выбранной программы

In [ ]:
prog = 'Экономика'
df_prog = df[df['program'] == prog]

plt.figure(figsize=(10,6))
plt.plot(df_prog['year'], df_prog['cost'], marker='o', label='Фактические данные')
plt.plot(pred_df['year'], pred_df['pred_cost'], marker='x', linestyle='--', label='Прогноз')
plt.title(f'Фактическая и прогнозная стоимость обучения: {prog}')
plt.xlabel('Год')
plt.ylabel('Стоимость, руб.')
plt.grid(True)
plt.legend()
plt.show()

## Выводы

- Построены модели линейной регрессии по образовательным программам.
- Выполнен анализ динамики стоимости обучения в МУИВ и вузах‑аналогах.
- Реализована функция прогноза стоимости обучения на основе исторических данных.
- Полученные результаты могут быть использованы в интеллектуальной системе мониторинга динамики стоимости образовательных услуг.